In [1]:
import plotly.graph_objects as go
import numpy as np
from phd_visualizations.constants import color_palette


# Create figure
fig = go.Figure()

# Sample data
yref = np.arange(10, 40, 2)
ymod = np.arange(10, 40, 2) + np.random.normal(0, 1, len(yref))
x = yref
Qc = [5, 10, 15, 20, 25] * (len(x)//5)
Qdc = [3, 8, 5, 10, 12] * (len(x)//5)
Qwct = np.array(Qc) - np.array(Qdc)
Tamb = [10, 15, 20, 25, 30] * (len(x)//5)

Tamb_min, Tamb_max = 10, 30
Qc_min, Qc_max = 5, 25

# Parameters
min_size = 0.4  # Minimum size of the donut hole
max_size = 1  # Maximum size of the donut hole
ring_width = 0.8  # Width of the donut ring

def polar_to_cartesian(r, theta_deg):
    # Helper: Convert polar to cartesian
    theta = np.radians(theta_deg)
    return cx + r * np.cos(theta), cy + r * np.sin(theta)

def lerp_color(c1, c2, t):
    # Linear interpolation between the two colors
    c1 = np.array([int(c1[i:i+2], 16) for i in (1, 3, 5)])
    c2 = np.array([int(c2[i:i+2], 16) for i in (1, 3, 5)])
    c = (1 - t) * c1 + t * c2
    return f"#{int(c[0]):02X}{int(c[1]):02X}{int(c[2]):02X}"

for idx in range(len(x)):
    cx, cy = x[idx], ymod[idx]  # Center

    # Map Qc[0] from [Qc_min, Qc_max] to [min_size, max_size] for r_inner
    r_inner = min_size + (Qc[idx] - Qc_min) / (Qc_max - Qc_min) * (max_size - min_size)
    # r_outer is 10% of min_size larger than r_inner
    r_outer = r_inner + ring_width * min_size
    max_radius = max_size + ring_width * min_size
    contribution_Qwct = Qwct[idx] / Qc[idx] * 360 
    theta_start = 0    # Starting angle of partial arc (in degrees)
    theta_end = contribution_Qwct    # Ending angle of partial arc

    # Generate arc path (SVG path syntax)
    num_points = 100
    theta = np.linspace(theta_start, theta_end, num_points)
    x_arc = cx + r_outer * np.cos(np.radians(theta))
    y_arc = cy + r_outer * np.sin(np.radians(theta))
    path = f"M{x_arc[0]},{y_arc[0]}" + ''.join([f"L{x},{y}" for x, y in zip(x_arc[1:], y_arc[1:])])
    path += f"L{cx},{cy}Z"  # Close the path back to the center

    # Add full outer circle (base)
    fig.add_shape(
        type="circle",
        x0=cx - r_outer, y0=cy - r_outer,
        x1=cx + r_outer, y1=cy + r_outer,
        fillcolor=color_palette["dc_green"],
        # layer="below",
        line=dict(width=0),  # Disable the line
        showlegend=True if idx == 0 else False,  # Show legend only for the first donut
        name="DC cooling power" if idx == 0 else "",
    )

    # Add partial arc segment (on top of base)
    fig.add_shape(
        type="path",
        path=path,
        fillcolor=color_palette["wct_purple"],
        line=dict(width=0),  # Disable the line
        showlegend=True if idx == 0 else False,  # Show legend only for the first donut
        name="WCT cooling power" if idx == 0 else "",
    )

    # Add inner filled circle (donut hole)
    
    # Interpolate color from cool_green to cool_red based on Tamb
    # Normalize Tamb to [0, 1]
    t_norm = (Tamb[idx] - Tamb_min) / (Tamb_max - Tamb_min)
    fillcolor = lerp_color("#ffffff", color_palette["plotly_yellow"], t_norm)
    
    fig.add_shape(
        type="circle",
        x0=cx - r_inner, y0=cy - r_inner,
        x1=cx + r_inner, y1=cy + r_inner,
        fillcolor=fillcolor,
        opacity=1,
        line_color="rgba(0, 0, 0, 0)",  # Transparent
    )

    # Add inner filled circle (donut hole)
    fig.add_shape(
        type="circle",
        x0=cx - max_radius, y0=cy - max_radius,
        x1=cx + max_radius, y1=cy + max_radius,
        fillcolor="rgba(0, 0, 0, 0)",  # Transparent
        line_color=color_palette["bg_gray"],
        line=dict(dash='dot'),
        showlegend=True if idx == 0 else False,  # Show legend only for the first donut
        name="Max. thermal load" if idx == 0 else "",
    )

# Add an additional (out of the range) weather circle for the legend
fig.add_shape(
    type="circle",
    x0=-9999,
    x1=-9999,
    fillcolor=color_palette["plotly_yellow"],
    opacity=1,
    line_color="rgba(0, 0, 0, 0)",  # Transparent
    showlegend=True,
    name="Weather scenario",
)

# Add invisible scatter trace to fix axis scaling
fig.add_trace(go.Scatter(
    x=yref, y=yref,
    mode="lines",
    marker_opacity=0,
    showlegend=True,
    name="Perfect fit",
))

# Layout
fig.update_layout(
    width=800,
    height=800,
    xaxis=dict(visible=True, range=[yref.min()*0.9, yref.max()*1.1]),
    yaxis=dict(visible=True, range=[yref.min()*0.9, yref.max()*1.1]),
    xaxis_title="Reference outputs",
    yaxis_title="Predicted outputs",
    margin=dict(t=100, b=10, l=10, r=10),  # Increased top margin for more space
    plot_bgcolor="white",
    title="<b>Output prediction validation</b><br>In the weather scenario, yellow indicates unfavorable weather conditions, white indicates favorable conditions.",
)

fig.show()


In [2]:
from phd_visualizations import save_figure

save_figure(
    fig,
    figure_name="cc-regression-validation-template",
    figure_path=".",
    formats=["png"],
)
    


2025-06-19 12:48:34.592 | INFO     | phd_visualizations:save_figure:42 - Figure saved in cc-regression-validation-template.png
